#### Gold: map-ready sections and daily summary
Combines the latest silver road sections with the milepost reference points to produce:
- `brp.gold.section_status`: every section with its status and **map line** (GeoJSON), one row per section per day
- `brp.gold.daily_summary`: miles open / reduced / closed and percentages, for the whole parkway and each state

**Linear referencing:** sections are stored by milepost (e.g. 63.5 to 63.9). Their positions are found by
interpolating between the whole-mile reference points, then each line runs:
start point → every whole milepost inside the section → end point.

**Steps**
1. **Settings:** imports, table names, schema
2. **Helpers + self-tests:** interpolate a milepost to coordinates; build a section's line
3. **Load:** latest silver day and the milepost reference points
4. **Build geometry:** one line per section
5. **Quality checks:** every line valid, inside the parkway area, and connected to its neighbours
6. **Write sections:** replace this day's rows (safe to re-run)
7. **Daily summary:** totals by scope, written the same way
8. **Review:** summary table and map

In [0]:
# Standard library
import bisect
import json

# Third-party
import pandas as pd
from pyspark.sql.types import (
    StructType, StructField, DateType, StringType, IntegerType, DoubleType, BooleanType, TimestampType
)

SILVER_TABLE    = "brp.silver.road_sections_status"
MILEPOSTS_TABLE = "brp.silver.mileposts"
GOLD_SECTIONS   = "brp.gold.section_status"
GOLD_SUMMARY    = "brp.gold.daily_summary"

GOLD_SECTIONS_SCHEMA = StructType([
    StructField("run_date",         DateType()),
    StructField("fetched_at",       TimestampType()),
    StructField("nps_as_of",        StringType()),
    StructField("state",            StringType()),
    StructField("section_order",    IntegerType()),
    StructField("mp_start",         DoubleType()),
    StructField("mp_end",           DoubleType()),
    StructField("length_mi",        DoubleType()),
    StructField("crossroads",       StringType()),
    StructField("status",           StringType()),
    StructField("status_raw",       StringType()),
    StructField("notes",            StringType()),
    StructField("gated",            BooleanType()),
    StructField("point_count",      IntegerType()),
    StructField("geometry_geojson", StringType()),
])

In [0]:
# ---------------------------------------------------------------------------
# HELPER FUNCTIONS: LINEAR REFERENCING
# Turn a milepost number into map coordinates by interpolating between the
# whole-mile reference points, and build a section's line from them.
# Coordinates are [longitude, latitude], the GeoJSON order.
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# LINEAR REFERENCING, PART 1: build_milepost_index()
# Puts the reference points in mile order, so the other helpers can find the
# neighbours of any milepost quickly (with a binary search).
# ---------------------------------------------------------------------------
def build_milepost_index(mileposts):
    """Sort [(milepost, lon, lat), ...] and keep a separate list of milepost numbers."""
    # bisect (below) needs a sorted list of plain numbers to search quickly
    points = sorted(mileposts)
    return [p[0] for p in points], points


# ---------------------------------------------------------------------------
# LINEAR REFERENCING, PART 2: coord_at()
# "Where is milepost 63.5 on the map?" We only know whole miles, so we find
# the two neighbours (mile 63 and mile 64) and take a proportional step
# between them — like finding a point halfway along a ruler segment.
# ---------------------------------------------------------------------------
def coord_at(mp, index):
    """Return [lon, lat] for any milepost, e.g. 63.5, by interpolation."""
    mps, points = index

    # Before the first or after the last reference point: use the end point
    if mp <= mps[0]:
        return [points[0][1], points[0][2]]
    if mp >= mps[-1]:
        return [points[-1][1], points[-1][2]]

    # bisect_right finds where mp would be inserted in the sorted list,
    # so points[i - 1] is the reference point just before mp and points[i] just after
    i = bisect.bisect_right(mps, mp)
    m0, lon0, lat0 = points[i - 1]
    m1, lon1, lat1 = points[i]

    # t = how far mp is between the two points (0 = at m0, 1 = at m1),
    # e.g. 63.5 between 63 and 64 -> t = 0.5 (halfway)
    t = (mp - m0) / (m1 - m0)
    return [round(lon0 + t * (lon1 - lon0), 6),
            round(lat0 + t * (lat1 - lat0), 6)]


# ---------------------------------------------------------------------------
# LINEAR REFERENCING, PART 3: section_line()
# "Draw the piece of road between milepost 0.5 and 1.5."
# A straight jump from start to end would cut corners on a curvy road, so
# the line includes every whole reference point in between:
#     [start point, whole miles inside, end point]
# ---------------------------------------------------------------------------
def section_line(mp_start, mp_end, index):
    """Coordinates for a section: start point, every whole milepost inside it, end point."""
    mps, points = index
    inner = [[lon, lat] for m, lon, lat in points if mp_start < m < mp_end]
    return [coord_at(mp_start, index), *inner, coord_at(mp_end, index)]


# ---------------------------------------------------------------------------
# SELF-TESTS (on a tiny made-up 3-point road)
# ---------------------------------------------------------------------------
test_index = build_milepost_index([(0, -80.0, 37.0), (1, -80.1, 36.9), (2, -80.2, 36.8)])

assert coord_at(0.5, test_index) == [-80.05, 36.95], "Self-test: halfway between 0 and 1"
assert coord_at(1, test_index) == [-80.1, 36.9], "Self-test: exactly on a reference point"
assert coord_at(-1, test_index) == [-80.0, 37.0], "Self-test: before the start uses the first point"
assert section_line(0.5, 1.5, test_index) == [[-80.05, 36.95], [-80.1, 36.9], [-80.15, 36.85]], \
    "Self-test: line includes the whole milepost inside it"
assert section_line(0.2, 0.8, test_index) == [[-80.02, 36.98], [-80.08, 36.92]], \
    "Self-test: short section between two reference points"

print("Helper self-tests passed")

In [0]:
latest = spark.sql(f"""
    SELECT MAX(fetched_at) AS d
    FROM {SILVER_TABLE}
""").first()["d"]
display(latest)

In [0]:
# ---------------------------------------------------------------------------
# LOAD THE LATEST SILVER DOWNLOAD AND THE MILEPOST REFERENCE POINTS
# Gold needs two inputs:
#   1. the road sections from the most recent download in silver
#   2. the milepost points, used to draw each section as a line on the map
# ---------------------------------------------------------------------------


# --- Step 1: Find the most recent download in silver ---
# Use fetched_at (exact download time), not run_date: there can be several
# downloads per day.
# .first() returns the single result row; ["d"] takes its value.
latest = spark.sql(f"""
    SELECT MAX(fetched_at) AS d
    FROM {SILVER_TABLE}
""").first()["d"]

# MAX() returns NULL (None in Python) if the table has no rows.
assert latest is not None, f"{SILVER_TABLE} is empty: run the silver notebook first"


# --- Step 2: Load that download's sections ---
# Keep only rows from the latest download, in page order (north to south).
# .toPandas() pulls the result out of Spark into a pandas table, which is
# easier to loop over and fine at this size (about 30 rows).
sections_pdf = spark.sql(f"""
    SELECT *
    FROM {SILVER_TABLE}
    WHERE fetched_at = TIMESTAMP '{latest}'
    ORDER BY section_order
""").toPandas()

# All rows come from one download, so they share the same run_date and fetched_at.
run_date   = sections_pdf["run_date"].iloc[0]
fetched_at = sections_pdf["fetched_at"].iloc[0]


# --- Step 3: Load the milepost reference points ---
# .collect() brings every row back to Python as a list of Row objects
# (about 470 rows, one per milepost).
milepost_rows = spark.sql(f"""
    SELECT milepost, longitude, latitude
    FROM {MILEPOSTS_TABLE}
""").collect()

# An empty list means the mileposts notebook hasn't been run yet.
assert milepost_rows, f"{MILEPOSTS_TABLE} is empty: run 04_reference_mileposts first"


# --- Step 4: Build the milepost lookup index ---
# Turn each Row into a plain (milepost, longitude, latitude) tuple, then
# build the index once. Every section reuses it to find its start point,
# end point and the whole mileposts in between.
index = build_milepost_index(
    [(r["milepost"], r["longitude"], r["latitude"]) for r in milepost_rows]
)

print(f"Download: {fetched_at} (run date {run_date}) | "
      f"Sections: {len(sections_pdf)} | Reference points: {len(milepost_rows)}")

In [0]:
# ---------------------------------------------------------------------------
# BUILD A MAP LINE FOR EVERY SECTION
# Each line is stored as GeoJSON text, e.g.
#   {"type": "LineString", "coordinates": [[-79.1, 37.8], [-79.12, 37.79], ...]}
# ---------------------------------------------------------------------------

gold_records = []
for s in sections_pdf.to_dict("records"):
    coords = section_line(s["mp_start"], s["mp_end"], index)
    gold_records.append({
        "run_date":         s["run_date"],
        "fetched_at":       s["fetched_at"],
        "nps_as_of":        s["nps_as_of"],
        "state":            s["state"],
        "section_order":    int(s["section_order"]),
        "mp_start":         s["mp_start"],
        "mp_end":           s["mp_end"],
        "length_mi":        s["length_mi"],
        "crossroads":       s["crossroads"],
        "status":           s["status"],
        "status_raw":       s["status_raw"],
        "notes":            s["notes"],
        "gated":            bool(s["gated"]),
        "point_count":      len(coords),
        "geometry_geojson": json.dumps({"type": "LineString", "coordinates": coords}),
        "_coords":          coords,          # kept only for the quality checks; not saved
    })

print(len(gold_records), "section lines built")

In [0]:
# ---------------------------------------------------------------------------
# QUALITY CHECKS ON THE GEOMETRY
# ---------------------------------------------------------------------------

MIN_LON, MAX_LON = -83.4, -78.7     # parkway area (same box as the milepost checks)
MIN_LAT, MAX_LAT = 35.2, 38.1
JOIN_TOLERANCE = 0.0001             # ~10 m: how close neighbouring sections' ends must be

# 1. One gold row per silver section
assert len(gold_records) == len(sections_pdf), "Check 1: section count changed"

# 2. Every line has at least a start and an end point
short = [g for g in gold_records if g["point_count"] < 2]
assert not short, f"Check 2: {len(short)} lines have fewer than 2 points"

# 3. Every coordinate is inside the parkway area
outside = [(g["mp_start"], c) for g in gold_records for c in g["_coords"]
           if not (MIN_LON <= c[0] <= MAX_LON and MIN_LAT <= c[1] <= MAX_LAT)]
assert not outside, f"Check 3: {len(outside)} coordinates outside the parkway area, e.g. {outside[:3]}"

# 4. Sections join up: each line's end is where the next line starts
ordered = sorted(gold_records, key=lambda g: g["mp_start"])
breaks = []
for prev, curr in zip(ordered, ordered[1:]):
    (x1, y1), (x2, y2) = prev["_coords"][-1], curr["_coords"][0]
    same_place = abs(x1 - x2) <= JOIN_TOLERANCE and abs(y1 - y2) <= JOIN_TOLERANCE
    if not same_place and abs(curr["mp_start"] - prev["mp_end"]) < 0.01:
        breaks.append(f"MP {prev['mp_end']} -> {curr['mp_start']}")
assert not breaks, f"Check 4: lines don't connect at {breaks}"

print("Geometry checks 1-4 passed")

In [0]:
# ---------------------------------------------------------------------------
# WRITE GOLD SECTIONS (SAFE TO RE-RUN)
# Replaces only this run_date's rows, like silver.
# ---------------------------------------------------------------------------

# pandas Timestamps pulled in via .toPandas() are timezone-naive, but Spark's
# Arrow conversion needs a timezone-aware value for a timestamp column.
# fetched_at was stored in UTC, so localize it to UTC to preserve the instant.
rows = []
for g in gold_records:
    row = []
    for field in GOLD_SECTIONS_SCHEMA.fields:
        value = g[field.name]
        if field.dataType == TimestampType() and isinstance(value, pd.Timestamp) and value.tzinfo is None:
            value = value.tz_localize("UTC").to_pydatetime()
        row.append(value)
    rows.append(tuple(row))

gold_df = spark.createDataFrame(rows, schema=GOLD_SECTIONS_SCHEMA)

(gold_df.write
    .mode("overwrite")
    .option("replaceWhere", f"run_date = '{run_date}'")
    .saveAsTable(GOLD_SECTIONS))

print(f"Wrote {gold_df.count()} sections to {GOLD_SECTIONS} for {run_date}")

In [0]:
# ---------------------------------------------------------------------------
# DAILY SUMMARY: TOTALS FOR THE WHOLE PARKWAY AND EACH STATE
# GROUPING SETS computes two groupings at once:
#   (run_date)        -> one row for the whole parkway (scope = 'ALL')
#   (run_date, state) -> one row per state (scope = 'VA' / 'NC')
# CAST(... AS INT) matches the table's INT column (COUNT returns a big integer).
# ---------------------------------------------------------------------------

summary_df = spark.sql(f"""
    SELECT
      run_date,
      COALESCE(state, 'ALL')                                                         AS scope,
      CAST(COUNT(*) AS INT)                                                          AS sections,
      ROUND(SUM(length_mi), 1)                                                       AS miles_total,
      ROUND(SUM(CASE WHEN status = 'open'    THEN length_mi ELSE 0 END), 1)          AS miles_open,
      ROUND(SUM(CASE WHEN status = 'reduced' THEN length_mi ELSE 0 END), 1)          AS miles_reduced,
      ROUND(SUM(CASE WHEN status = 'closed'  THEN length_mi ELSE 0 END), 1)          AS miles_closed,
      ROUND(SUM(CASE WHEN status = 'unknown' THEN length_mi ELSE 0 END), 1)          AS miles_unknown,
      ROUND(100 * SUM(CASE WHEN status = 'open' THEN length_mi ELSE 0 END)
                / SUM(length_mi), 1)                                                 AS pct_open,
      ROUND(100 * SUM(CASE WHEN status IN ('open', 'reduced') THEN length_mi ELSE 0 END)
                / SUM(length_mi), 1)                                                 AS pct_passable,
      MAX(nps_as_of)                                                                 AS nps_as_of
    FROM {GOLD_SECTIONS}
    WHERE run_date = '{run_date}'
    GROUP BY GROUPING SETS ((run_date), (run_date, state))
""")

(summary_df.write
    .mode("overwrite")
    .option("replaceWhere", f"run_date = '{run_date}'")
    .saveAsTable(GOLD_SUMMARY))

display(summary_df)

In [0]:
%pip install folium

In [0]:
# ---------------------------------------------------------------------------
# REVIEW: DRAW THE SECTIONS ON A MAP, COLORED BY STATUS
# Uses folium (a Python wrapper for Leaflet, the same library the public
# dashboard will use). Hover a line to see its mileposts and status.
# ---------------------------------------------------------------------------
import folium   # if missing: add a cell at the top with  %pip install folium

COLORS = {"open": "#2f6b4f", "reduced": "#d68910", "closed": "#c0392b", "unknown": "#7f8c8d"}

road_map = folium.Map(location=[36.6, -80.8], zoom_start=7, tiles="OpenStreetMap")
for g in gold_records:
    folium.GeoJson(
        json.loads(g["geometry_geojson"]),
        style_function=lambda _, color=COLORS[g["status"]]: {"color": color, "weight": 5},
        tooltip=f"{g['state']} MP {g['mp_start']}–{g['mp_end']}: {g['status']} ({g['status_raw']})",
    ).add_to(road_map)

road_map   # the map appears below the cell

In [0]:
%sql
SELECT * FROM brp.gold.section_status